In [ ]:
from pathlib import Path
import sys
import subprocess

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

DATA_PATH = ROOT / "single-cell-tracks_exp1-6_noErbB2.csv.gz"
META_PATH = ROOT / "01-readme-experiment-description_2022-04-05.csv"
SCRIPT_PATH = ROOT / "scripts" / "compare_spatiotemporal_behavior_copy.py"
OUTPUT_ROOT = ROOT / "analysis_outputs_A1"

OUTPUT_ROOT.mkdir(exist_ok=True)

print("ROOT:", ROOT)
print("DATA exists:", DATA_PATH.exists())
print("META exists:", META_PATH.exists())
print("SCRIPT exists:", SCRIPT_PATH.exists())
print("OUTPUT_ROOT:", OUTPUT_ROOT)

In [ ]:
cmd = [
    sys.executable,
    str(SCRIPT_PATH),
    "--data-path", str(DATA_PATH),
    "--meta-path", str(META_PATH),
    "--signal-col", "ERKKTR_ratio",
    "--group-by", "mutation",
    "--spatial-radius", "60",
    "--future-window-frames", "3",
    "--jump-quantile", "0.9",
    "--include-mutations",
    "WT",
    "AKT1_E17K",
    "PIK3CA_E545K",
    "PIK3CA_H1047R",
    "PTEN_del",
    "--output-dir", str(OUTPUT_ROOT),
]

print("Running:")
print(" ".join(cmd))

result = subprocess.run(cmd, capture_output=True, text=True)

print("RETURN CODE:", result.returncode)
print("\nSTDOUT:")
print(result.stdout)
print("\nSTDERR:")
print(result.stderr)

if result.returncode != 0:
    raise RuntimeError("A1 script failed. Check STDERR above.")

In [ ]:
#wczytanie wyników
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import mannwhitneyu

A1_DIR = ROOT / "analysis_outputs_A1" / "comparison_mutation_ERKKTR_ratio"
OUTPUT_DIR = ROOT / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

group = pd.read_csv(A1_DIR / "group_level_summary.csv")
block = pd.read_csv(A1_DIR / "block_level_summary.csv")

display(group)
display(block.head())

print("group columns:", group.columns.tolist())
print("block columns:", block.columns.tolist())

In [ ]:
#średnie RR i St
wanted = ["WT", "AKT1_E17K", "PIK3CA_E545K", "PIK3CA_H1047R", "PTEN_del"]

rr_stats = (
    block[block["comparison_group"].isin(wanted)]
    .groupby("comparison_group")["relative_risk"]
    .agg(["mean", "std", "count"])
    .reset_index()
)

rr_stats["std_error"] = rr_stats["std"] / np.sqrt(rr_stats["count"])

rr_stats = rr_stats.rename(columns={
    "comparison_group": "mutation",
    "mean": "mean_RR"
})

rr_stats

In [ ]:
wt_values = block.loc[block["comparison_group"] == "WT", "relative_risk"].dropna()

test_rows = []

for mut in ["AKT1_E17K", "PIK3CA_E545K", "PIK3CA_H1047R", "PTEN_del"]:
    mut_values = block.loc[block["comparison_group"] == mut, "relative_risk"].dropna()
    
    stat, p = mannwhitneyu(wt_values, mut_values, alternative="two-sided")
    p_corr = min(p * 4, 1.0)
    
    test_rows.append({
        "mutation": mut,
        "p_value_vs_WT": p_corr,
        "significant": p_corr < 0.05
    })

stats_table = pd.DataFrame(test_rows)
stats_table

In [ ]:
final_table = rr_stats[["mutation", "mean_RR", "std_error"]].merge(
    stats_table,
    on="mutation",
    how="left"
)

final_table.loc[final_table["mutation"] == "WT", "p_value_vs_WT"] = np.nan
final_table.loc[final_table["mutation"] == "WT", "significant"] = False

order = ["WT", "AKT1_E17K", "PIK3CA_E545K", "PIK3CA_H1047R", "PTEN_del"]
final_table["mutation"] = pd.Categorical(final_table["mutation"], categories=order, ordered=True)
final_table = final_table.sort_values("mutation")

final_table.to_csv(OUTPUT_DIR / "mutations_comparison_table.csv", index=False)

final_table

In [ ]:
plot_data = final_table.copy()

plt.figure(figsize=(8, 5))
plt.bar(
    plot_data["mutation"].astype(str),
    plot_data["mean_RR"],
    yerr=plot_data["std_error"],
    capsize=5
)

plt.axhline(1, linestyle="--", linewidth=1)
plt.ylabel("Mean Relative Risk")
plt.xlabel("Mutation")
plt.title("ERK spatiotemporal propagation across PI3K-AKT mutations")
plt.xticks(rotation=30, ha="right")

for i, row in plot_data.reset_index(drop=True).iterrows():
    if bool(row["significant"]):
        plt.text(
            i,
            row["mean_RR"] + row["std_error"] + 0.05,
            "*",
            ha="center",
            va="bottom",
            fontsize=16
        )

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "mutations_barplot.png", dpi=300)
plt.show()

In [ ]:
print((OUTPUT_DIR / "mutations_comparison_table.csv").exists())
print((OUTPUT_DIR / "mutations_barplot.png").exists())

In [ ]:
print("SCRIPT_PATH:", SCRIPT_PATH)
print("SCRIPT exists:", SCRIPT_PATH.exists())

with open(SCRIPT_PATH, "r", encoding="utf-8") as f:
    txt = f.read()

print("--include-mutations" in txt)
print("A1 mutation comparison" in txt)
print("if __name__ == '__main__':" in txt or 'if __name__ == "__main__":' in txt)